<a href="https://colab.research.google.com/github/mkane968/digital-text-methods/blob/main/Tutorial_8_Machine_Classification_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 8: Machine Classification with Logistic Regression

**Machine classification** asks a different kind of question from topic modeling.

In topic modeling, we do not provide categories in advance.

In **supervised classification**, we begin with examples that already have labels and train a model to predict those labels for new examples.

In this tutorial, we will build a simple **logistic regression text classifier**.

### In this tutorial, we will:
- work with labeled textual data
- divide data into training and test sets
- convert text into numerical features
- train a logistic regression classifier
- make predictions
- evaluate accuracy and errors
- consider what a classifier has actually learned


## 1. Create a Small Labeled Dataset

For this demonstration, imagine that we want to classify short passages into two categories:

- `nature`
- `city`

The labels are already known. These labeled examples are our **training data**.


In [1]:
import pandas as pd

texts = [
    "flowers bloom beside the quiet river",
    "trees cover the green hills",
    "birds fly above the forest",
    "the river moves through the valley",
    "leaves fall beneath the trees",
    "the field is covered with flowers",
    "wind moves through the forest trees",
    "birds rest beside the river",
    "green plants grow across the field",
    "the quiet valley lies beneath the hills",
    "traffic fills the crowded street",
    "people walk past tall buildings",
    "cars move through the city",
    "shops line the busy street",
    "crowds gather near the station",
    "the building rises above the traffic",
    "people cross the crowded avenue",
    "cars wait beside the city buildings",
    "the street is filled with shops",
    "traffic moves past the station"
]

labels = [
    "nature","nature","nature","nature","nature",
    "nature","nature","nature","nature","nature",
    "city","city","city","city","city",
    "city","city","city","city","city"
]

df = pd.DataFrame(
    list(zip(texts, labels)),
    columns=["text", "label"]
)

df


,text,label
0,flowers bloom beside the quiet river,nature
1,trees cover the green hills,nature
2,birds fly above the forest,nature
3,the river moves through the valley,nature
4,leaves fall beneath the trees,nature
5,the field is covered with flowers,nature
6,wind moves through the forest trees,nature
7,birds rest beside the river,nature
8,green plants grow across the field,nature
9,the quiet valley lies beneath the hills,nature


In a real research project, labels might represent:

- genres
- disciplines
- rhetorical moves
- sentiment categories
- document types
- manually coded textual features

The quality and meaning of the classifier depend heavily on the quality and meaning of these labels.


## 2. Training Data and Test Data

We should not evaluate a classifier using the exact same examples it learned from.

Instead, we divide our labeled data into:

- **training data** — examples used to learn patterns
- **test data** — examples held back to evaluate predictions


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))


Training examples: 14
Test examples: 6


Here, 30% of the examples are held out for testing.

`random_state=42` makes the split reproducible.

`stratify` helps preserve the balance of our categories in both sets.


## 3. Convert Text into Features

Logistic regression cannot directly interpret sentences.

We will represent the texts using **TF-IDF**, which gives words numerical weights based on their occurrence in documents.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")

X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)


Notice an important distinction:

We use `.fit_transform()` on the **training data**, but only `.transform()` on the **test data**.

The vectorizer learns its vocabulary from the training set. The test set remains unseen during training.


## 4. Train the Classifier

Now we can train a logistic regression model:


In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_vectors, y_train)


LogisticRegression()

The model examines the relationship between the textual features and the known labels in the training data.

It learns which features help distinguish the categories.


## 5. Make Predictions

Now ask the model to predict the labels of the held-out test texts:


In [5]:
predictions = model.predict(X_test_vectors)

predictions


array(['city', 'nature', 'city', 'nature', 'nature', 'city'], dtype=object)

Compare the predicted labels with the actual labels:


In [6]:
results = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": predictions
})

results


,text,actual,predicted
14,crowds gather near the station,city,city
9,the quiet valley lies beneath the hills,nature,nature
13,shops line the busy street,city,city
6,wind moves through the forest trees,nature,nature
2,birds fly above the forest,nature,nature
16,people cross the crowded avenue,city,city


A correct prediction means the predicted label matches the human-provided label.

An incorrect prediction is especially useful to inspect: it may reveal ambiguity in the text, limitations of the features, problems with the labels, or insufficient training data.


## 6. Accuracy

One simple evaluation measure is **accuracy**:

> What proportion of the test examples did the model classify correctly?


In [7]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

accuracy


1.0

An accuracy of `1.0` means 100% of the test examples were classified correctly.

But accuracy should always be interpreted in context. This toy dataset contains very obvious categories and very little linguistic complexity. Real classification tasks are substantially harder.


## 7. Confusion Matrix

A **confusion matrix** shows which categories the model predicted correctly and which it confused.


In [8]:
from sklearn.metrics import confusion_matrix

labels_order = ["nature", "city"]

cm = confusion_matrix(
    y_test,
    predictions,
    labels=labels_order
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual nature", "Actual city"],
    columns=["Predicted nature", "Predicted city"]
)

cm_df


,Predicted nature,Predicted city
Actual nature,3,0
Actual city,0,3


The diagonal cells represent correct classifications. The other cells represent errors.

For a research project, examining **which categories are confused with one another** can be as informative as the overall accuracy score.


## 8. Try the Classifier on New Text

Once trained, the classifier can make predictions for new examples:


In [9]:
new_texts = [
    "birds gather beside the green river",
    "traffic moves between tall buildings"
]

new_vectors = vectorizer.transform(new_texts)

model.predict(new_vectors)


array(['nature', 'city'], dtype=object)

Try replacing these sentences with your own examples.

What happens if you create a deliberately ambiguous sentence containing vocabulary from both categories?


## 9. What Words Did the Model Use?

One advantage of logistic regression is that we can inspect which features were associated with its classifications.

First, retrieve the feature names:


In [10]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

feature_weights = pd.DataFrame({
    "word": feature_names,
    "weight": coefficients
})

feature_weights.sort_values("weight").head(10)


,word,weight
31,traffic,-0.497688
6,city,-0.418451
5,cars,-0.418451
29,street,-0.371640
20,past,-0.341152
4,buildings,-0.318766
27,shops,-0.232763
12,filled,-0.232763
3,building,-0.232700
25,rises,-0.232700


Now look at the other end:


In [11]:
feature_weights.sort_values("weight", ascending=False).head(10)


,word,weight
26,river,0.517705
11,field,0.367779
14,flowers,0.365135
32,trees,0.341667
15,green,0.339632
33,valley,0.251914
8,covered,0.229493
1,birds,0.227970
24,rest,0.227970
10,fall,0.197784


The direction of the weights corresponds to the order of the model's categories:


In [12]:
model.classes_


array(['city', 'nature'], dtype=object)

These weights can help us investigate **what textual signals the model relied on**.

They should not automatically be interpreted as the defining characteristics of a category. They describe patterns learned from this particular training dataset.


## 10. Classification as a Research Workflow

A basic supervised text-classification workflow looks like this:

**Labeled Texts → Train/Test Split → Numerical Features → Train Model → Predict → Evaluate → Inspect Errors**

The most important point is that the classifier learns from **examples supplied by humans**.

If the training data are small, inconsistent, biased, or poorly labeled, those limitations affect the resulting model.


## Try It Yourself

1. Run the classifier as written.
2. Examine the test-set predictions.
3. Write two new `nature` examples and two new `city` examples.
4. Ask the classifier to predict them.
5. Create one deliberately ambiguous example.
6. Inspect the highest-weighted words.

Then respond in a text cell:

**What seems to distinguish the two categories for this classifier? Where might the model fail if it were applied to a larger or more complex corpus?**


## From Classification to Your Research

Classification can be useful when you have a set of texts that have already been **coded or categorized** and want to investigate whether those categories can be predicted from textual patterns.

Possible applications include:

- identifying genres
- classifying rhetorical moves
- predicting document categories
- examining disciplinary differences
- scaling a manually coded scheme to a larger corpus

A classifier does not independently discover what categories mean. Its predictions depend on:

- the categories researchers define
- the examples used for training
- the textual features provided to the model
- the evaluation procedure

For that reason, evaluation should include not only **how often the model is correct**, but also **what kinds of errors it makes and why**.


## References and Additional Resources

- **scikit-learn: Working with Text Data**  
  https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html

- **scikit-learn: Logistic Regression**  
  https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

- **scikit-learn: TF-IDF Vectorizer**  
  https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

- **Introduction to Cultural Analytics & Python — Melanie Walsh**  
  https://melaniewalsh.github.io/Intro-Cultural-Analytics/


**AI Use Disclosure:** This tutorial was developed by the instructor with assistance from AI tools for drafting and refining explanations, examples, and Python code. All materials were reviewed, edited, and adapted by the instructor for this course.